In [8]:
import numpy as np
import pandas as pd
import scipy

In [9]:
"""
SHOPPING LIST:
- Function that integrates cos^n over an area of the unit sphere (pyramidal, see sketch)
- Function that returns the expected count rate
- I/O reader, then scipy.curve_fit for n in the above:
    x Read the CSV files
    x Convert angles to radian
    x Determine the opening angle based on inter-detector distance
    x 
"""

"""
SPECIFICATIONS:
Our length units MUST be cm. Every variable must give units as such.
Our time units may vary and MUST be specified in the docstring. This is to prevent floating-point annihilation issues.
TODO: Specify time units in all functions
"""

'\nSPECIFICATIONS:\nOur length units MUST be cm. Every variable must give units as such.\nOur time units may vary and MUST be specified in the docstring. This is to prevent floating-point annihilation issues.\nTODO: Specify time units in all functions\n'

In [15]:
SIDE_LENGTH = 5
PLANE_DIST = 10


def area_integral(n: float = 2.0, theta: float = 0, distance: float = PLANE_DIST):
    """
    For a cos^{n} distribution over the unit sphere, return the expected count rate of two planar detectors in parallel with distance {distance} between them.

    The count rate for two points in a line: Consider an area dA1 on the lower plane at x1, and an area dA2 on the upper plane at x2. Then a line from the atmosphere through both areas sees the lower area as d/r dA1; dA2 as seen from x1 subtends the solid angle (d/r) dA2 / r^2. Thus, we need to scale cos^n by d**2/r**4.

    The area arguments only depend on the relative offsets, so we can simply double-integrate over the x- and y-offsets.
    TODO change to Gauss-Legendre quadrature. https://stackoverflow.com/questions/77002315/get-2d-gauss-legendre-quadrature-points-and-weights-using-numpy .
    """
    a = SIDE_LENGTH
    d = distance

    def integrand(dy, dx):
        return (
            (a - np.abs(dx))
            * (a - np.abs(dy))
            * (d / (dx**2 + dy**2 + d**2)) ** 2
            * np.cos(theta) ** n
        )

    return scipy.integrate.dblquad(integrand, -a, a, -a, a)[0]

In [11]:
WINDOW = 30e-3  # 30 us = 0.03 ms = 30e-6 s


def accidental_rate(singles_rate=0.5, window=WINDOW):
    """
    Returns the rate of accidentals from both detectors as given in the source.
    singles_rate must have the same unit as WINDOW.
    Returns units of WINDOW^{-1}
    """
    return 2 * singles_rate**2 * window

In [12]:
ACCIDENTALS = 0.1
INTENSITY = 170  # Per HOUR


def count_rate(
    n: float = 2,
    intensity: float = 170,  # In units of HOUR^-1
    accidentals: float = ACCIDENTALS,  # In units of HOUR^-1
    timespan: float = 1,  # In units of HOUR
    theta: float = 0,
    distance: float = 10,
):
    """
    Returns the expected count rate for an exponent {n}, maximal {intensity} of only muons and {accidental} rate over a given {timespan} in HOURS, when measuring over {theta} with {distance} between planes.
    """
    return timespan * (intensity * area_integral(n, theta, distance) + accidentals)

In [13]:
def read_one(filename: str):
    """
    Reads a single filename. Obtains angle theta and count.
    """
    data = pd.read_csv(
        filename,
        delimiter=" ",
        header=5,
        index_col=0,
        names=["time", "adc", "sipm", "deadtime", "temp"],
    )  # Time in ms
    return data


def read_all(folder: str):
    """
    TODO
    Reads all files. Obtains distance, etc.
    """

In [16]:
sample = read_one("data/examples/Run2_S.TXT")
counts = len(sample.index)
# print(counts / sample.time.iloc[-1], accidental_rate(0.05))
SINGLES_RATE = scipy.optimize.least_squares(
    lambda x: accidental_rate(x) - counts / sample.time.iloc[-1], 0.05
).x
ACCIDENTALS = accidental_rate(SINGLES_RATE) * 1000 * 3600
print(ACCIDENTALS)

AttributeError: 'tuple' object has no attribute 'y'